# Mastering LightGBM
Welcome to the guide on LightGBM.
### Kaggle Dataset:
[Titanic Dataset](https://www.kaggle.com/c/titanic)


### Why and What: Install Package
**WHY:** Required library.
**WHAT:** pip install.


In [ ]:
!pip install lightgbm -q


### Why and What: Imports
**WHY:** Libraries for data manipulation.
**WHAT:** pandas, numpy, sklearn, matplotlib.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')


## Part 1: Theory Recap

LightGBM focuses on immense speed and memory efficiency for massive datasets.

### Real-World Analogy
Instead of measuring every single grain of sand to split them (exact greedy), you sweep the sand into 256 distinct buckets (histograms). Then you only ever evaluate splits between buckets. It is vastly faster and almost as accurate.

### Core Mathematics

1. **Histogram Binning Variance Gain:**
$$ V_j(d) = \frac{1}{n} \left( \frac{(\sum_{x_i \in A_l} g_i)^2}{n_l^j(d)} + \frac{(\sum_{x_i \in A_r} g_i)^2}{n_r^j(d)} \right) $$
* $V_j(d)$ : Variance gain for feature $j$ split at bin $d$.
* $g_i$ : Gradients of the instances.
* $n_l, n_r$ : Count of instances falling into the left and right bins.
* *Note:* This avoids sorting all feature values, changing complexity from $O(N \log N)$ to $O(N)$.

2. **Leaf-Wise Tree Growth:**
$$ \text{Split Node} = \arg\max_{\text{leaves}} \Delta \text{Loss} $$
* Unlike XGBoost which grows level-by-level, LightGBM searches all current leaves and splits the one that will reduce the global loss the absolute most. This results in deeper, asymmetrical trees.


### Why and What: Loading Data
**WHY:** Real world data.
**WHAT:** Titanic.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
display(df.head())


### Why and What: Preprocessing
**WHY:** ML models need clean numerical data.
**WHAT:** Fillna, encode, scale.


In [ ]:
df_clean = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)
X = df_clean.drop('Survived', axis=1).values
y = df_clean['Survived'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


## Part 2: From Scratch Implementation
### Why and What: Scratch
**WHY:** Demystify the black box.
**WHAT:** Numpy implementation.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
class ScratchModel:
    def __init__(self, n=50, lr=0.1, bins=32):
        self.n=n; self.lr=lr; self.bins=bins; self.trees=[]; self.b=[]
    def fit(self, X, y):
        X_bin = np.zeros_like(X, dtype=int)
        for i in range(X.shape[1]):
            b = np.histogram_bin_edges(X[:, i], bins=self.bins)
            self.b.append(b)
            X_bin[:, i] = np.digitize(X[:, i], b) - 1
        pred = np.zeros(y.shape[0])
        for _ in range(self.n):
            p = 1 / (1 + np.exp(-pred))
            tree = DecisionTreeRegressor(max_depth=1).fit(X_bin, y - p)
            pred += self.lr * tree.predict(X_bin)
            self.trees.append(tree)
    def predict_proba(self, X):
        X_bin = np.zeros_like(X, dtype=int)
        for i in range(X.shape[1]):
            X_bin[:, i] = np.digitize(X[:, i], self.b[i]) - 1
        pred = np.zeros(X.shape[0])
        for t in self.trees: pred += self.lr * t.predict(X_bin)
        return 1 / (1 + np.exp(-pred))
    def predict(self, X): return (self.predict_proba(X) >= 0.5).astype(int)


### Why and What: Evaluation
**WHY:** Verify correctness.
**WHAT:** Predict and accuracy.


In [ ]:
scratch_model = ScratchModel()
scratch_model.fit(X_train, y_train)
y_pred_custom = scratch_model.predict(X_test)
print(f"Scratch Accuracy: {accuracy_score(y_test, y_pred_custom)*100:.2f}%")


## Part 3: Library Implementation
### Why and What: Library
**WHY:** Optimized production code.
**WHAT:** Official package.


In [ ]:
from lightgbm import LGBMClassifier
model_library = LGBMClassifier(n_estimators=50, random_state=42)
model_library.fit(X_train, y_train)
print('LightGBM Accuracy:', accuracy_score(y_test, model_library.predict(X_test)))


### Why and What: Visualizations
**WHY:** Diagnose behavior.
**WHAT:** ROC and Importances.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if hasattr(model_library, 'predict_proba'):
    fpr, tpr, _ = roc_curve(y_test, model_library.predict_proba(X_test)[:, 1])
    axes[0].plot(fpr, tpr, color='darkorange', label=f'ROC (area = {auc(fpr, tpr):.2f})')
axes[0].plot([0,1], [0,1], 'k--', label='Random Guess')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(loc='lower right')
importances = getattr(model_library, 'feature_importances_', None)
if importances is not None:
    idx = np.argsort(importances)
    axes[1].barh(range(len(idx)), importances[idx], color='teal', label='Importance')
    axes[1].set_yticks(range(len(idx)))
    axes[1].set_yticklabels(df_clean.drop('Survived', axis=1).columns[idx])
    axes[1].legend(loc='lower right')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Relative Importance'); axes[1].set_ylabel('Features')
plt.tight_layout()
plt.show()


## Part 4: Hyperparameter Experiments
### Why and What: Tuning
**WHY:** Optimize model.
**WHAT:** Grid search.


In [ ]:
res=[]
for lr in [0.01, 0.1]:
 for leaves in [15, 31, 63]:
  clf=LGBMClassifier(num_leaves=leaves, learning_rate=lr, random_state=42)
  res.append({'leaves':leaves, 'lr':lr, 'acc':cross_val_score(clf, X_scaled, y, cv=3).mean()})
sns.lineplot(data=pd.DataFrame(res), x='leaves', y='acc', hue='lr')
plt.title('LightGBM Hyperparameters'); plt.legend(title='LR'); plt.show()


## Part 5: Interview Corner
**Q: Leaf-wise vs Level-wise?**
**A:** LightGBM grows leaf-wise (best-first), reducing loss faster but more prone to overfitting than level-wise (XGBoost).


## Key Takeaways
- Histogram Binning.
- Leaf-wise growth.
- High speed.
- Low memory.
- Prone to overfitting on small data.
